In [ ]:
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel, EmailStr, Field, field_validator,
    model_validator, SecretStr, ValidationError,
)

VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

Nesse bloco, nota-se a importação das libs `hashlib`, para garantir a criptografia das senhas, e `re`, lib para o uso de **expressões regulares Regex**, usadas para criar padrões rigorosos nos textos. Podemos ver o uso de Regex logo abaixo com as constantes `VALID_PASSWORD_REGEX` e `VALID_NAME_REGEX`, que pré compilam regras (com o uso de `re.compile()`) para garantir que o Name e a Password possuam apenas caracteres válidos de acordo como foi estabelecido.

In [ ]:
class Role(enum.IntFlag):
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(
        default=None, description="The role of the user", examples=[1, 2, 4, 8]
    )

Agora, temos as classes Role e User. Na role, o autor decidiu por escrever as expressões binárias na mão (diferente do exemplo da aula anterior). 
Na User, a estrutura é exatamente a mesma do exemplo anterior (SecretStr, frozen=True, Field(), etc.).

In [ ]:
@field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

Uso do `field_validator` importado anteriormente para uma validação simples. Aqui, ele compara o valor "v" de Name com a expressão regular. Se fugir das regras estabelecidas (apenas letras), retorna `ValueError`.

In [ ]:
@field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

Em resumo, esse `field_validator` trata de converter as roles recebidas de forma inteligente. A `validate_role` recebe um valor que pode ser int, string ou um objeto já formatado. Diante disso, a função usa `lambda` para simplificar a sintaxe e tratar diferentes tipos de dados que podem ser recebidos. Por exemplo, o sistema pode receber ora "1" ora "Author" de acordo com que sistema alheio ele está conversando; ambos são a mesma coisa, mas precisam ser lidas e interpretadas de forma diferente. Essa função organiza isso de forma que todas as entradas possam ser lidas devidamente sem gerar erros.

In [ ]:
@model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

Esse trecho implementa uma espécie de validação global quanto aos dados recebidos. Em resumo, aqui é verificado inicialmente se o Name ou a Password existem em "v", se sim, verifica se o Name estaria contido dentro da Password, se não, utiliza a constante `VALID_PASSWORD_REGEX` para verificar se a Password está de acordo com as restrições estabelecidas.

Após isso, finalmente vemos o uso da `hashlib`. Em resumo, essa lib irá criptografar a senha para sha256 de forma unilateral (ou seja, por meio de uma operação que não pode ser revertida) e depois a transforma numa string legível. 

No fim, quando ocorrer o return, a chave Password irá ser destruída e realocada para abrigar a string contendo a senha encriptada, agora no campo SecretStr.

In [ ]:
    def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid:")
        print(e)

def main() -> None:
    test_data = dict(
        good_data={"name": "Arjan", ...},
        bad_role={"role": "Programmer", ...},
        bad_data={"email": "bad email", ...},
        bad_name={"name": "Arjan<-_->", ...},
        duplicate={"password": "Arjan123", ...},
        missing_data={"email": "<bad data>", ...},
    )

    for example_name, data in test_data.items():
        print(example_name)
        validate(data)
        print()

if __name__ == "__main__":
    main()

Por fim, o autor mockou todos os casos de erro possível para realizar os testes (fora o good_data, que é um caso que funciona) e mostrar a funcionalidade das funções Pydantic.